# Descriptive analysis of green & brown employment based on EU-LFS data
Felix Zaussinger | 26.08.2022, updated 04.10.2022

**Core Analysis Goal(s)**
1. Create industry and regional plots based on final GBN classification (weighted and unweighted)
2. Create those plots both in an absolute and relative way

**Key Insight(s)**
1. Absolute numbers for unweighted shares closely correspond to independent figures of green employment based on EGSS data
2.
3.

In [1]:
import os
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from src import utils
import mapping_career_causeways

# OPTIONAL: Load the "autoreload" extension so that code can change
%load_ext autoreload

# OPTIONAL: always reload modules so that as you change code in src, it gets loaded
%autoreload 2

# Settings
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

sns.set_style("ticks")
sns.set_context("paper", font_scale=1.5)
sns.set(rc={"figure.dpi": 300})
sns.set_style("ticks")

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 120)

logging.basicConfig(level=logging.INFO, stream=sys.stdout)

# load paths
useful_paths = utils.UsefulPaths()

KeyboardInterrupt: 

In [ ]:
# preprocessing flag
reprocess = False

In [ ]:
from src.data.framework import Esco

esco = Esco()

#### Prepare EU-LFS data

In [ ]:
from src.data.lfs import EuLfs
from src.data.framework import Classifications

# EU-LFS data
config = utils.load_config(os.path.join(useful_paths.config_dir, "eu_lfs_config.yml"))
lfs = EuLfs(config=config)
year = 2019

lfs_data = lfs.read_preprocessed_file(
    year=year,
    input_fname_lfs="eu_lfs_merged_{year}_with_final_unweighted_shares_and_earnings_incdecil_imputed",
)

In [ ]:
# add column for COAL (CPO Scenario)
# criteria for coal phase-out scenario
coal_occupations = {
    "Mining, manufacturing and construction supervisors": "312",
    "Mining and mineral processing plant operators": "811",
    "Manufacturing, mining, construction, and distribution managers": "132",
    "Mining and construction labourers": "931",
}
coal_industries = {
    "Electricity, Gas, Steam and Air Conditioning Supply": "D",
    "Mining and Quarrying": "B",
}

occ_subset = lfs_data["ISCO3D_label"].isin(list(coal_occupations.keys()))
ind_subset = lfs_data["NACE1D_label"].isin(list(coal_industries.keys()))

# use totals for COAL
lfs_data["COEFF_share_coal"] = lfs_data["COEFF"]

# mask based on criteria
lfs_data.loc[~(occ_subset & ind_subset), "COEFF_share_coal"] = np.nan

In [ ]:
# shorten NACE labels
nace_short = pd.read_csv(
    os.path.join(
        useful_paths.data_raw,
        "classifications",
        "NACE_REV2_1d_section_codes_short_names.csv",
    ),
    index_col=0,
)

nace_mapping = dict(zip(nace_short.NACE1D_label, nace_short.NACE1D_label_short))
# lfs_data = lfs_data.replace(to_replace={"NACE1D_label": nace_mapping})
nace_short.columns

In [ ]:
lfs_data = pd.merge(lfs_data, nace_short, on="NACE1D_label", how="left")

In [ ]:
lfs_data

In [ ]:
from src.visualization.visualize import EulfsVis

config_paths = "paths_config.yml"
config_data = "data_config.yml"
config_vis = "vis_config.yml"

# EU-LFS
eulfs_visualiser = EulfsVis(
    fn_config_data=config_data,
    fn_config_path=config_paths,
    fn_vis_config=config_vis,
    year=2019,
)

# assign updated data panel
eulfs_visualiser.data_panel = lfs_data


#### Incorporate publication guidelines by Eurostat (thresholds for sample sizes, etc.)

In all reports, including both published and unpublished papers, three thresholds related to cell size will be distinguished for LFS results:
- Confidentiality threshold - below three observations (unweighted sample), results must not be published;
- Reliability thresholds - regarding reliability restrictions, Eurostat defines two limits, called 'a' and 'b'. Reliability limits depend on the sample size and design in the individual Member States.
    - Estimates corresponding to a (weighted) population below limit 'a' must not be published;
    - Estimates corresponding to a (weighted) population between limit 'a' and limit ‘b’ may be published with a warning concerning their limited reliability. This applies to quarterly data, annual averages of quarterly data, yearly data and ad hoc module results. The limits vary across Member States, years, and type of dataset. The thresholds “a” and “b” are provided in the CSV files in the “datafileinfo” folders in the anonymised microdata releases. More information can be found also at the LFS Statistics Explained pages, section “Publication guidelines and thresholds”.

See: C:\eurostat_data\raw\Yearly_Data\YearlyFiles_83_2019\_YearlyFiles\datafileinfo

In [ ]:
threshold_a = 3

### Visualisation

Define cols to process and plot

In [ ]:
cols_to_calc = [
    "COEFF_share_green",
    "COEFF_share_coal",
    "COEFF_share_brown_sl",
    "COEFF_share_brown_slt",
]

cols_to_aggregate = [
    "COEFF",
    "COEFF_share_green",
    "COEFF_share_coal",
    "COEFF_share_brown_sl",
    "COEFF_share_brown_slt",
]

cols_to_plot = [
    "COEFF_share_green_relative",
    "COEFF_share_coal_relative",
    "COEFF_share_brown_sl_relative",
    "COEFF_share_brown_slt_relative",
    "COEFF_share_green",
    "COEFF_share_coal",
    "COEFF_share_brown_sl",
    "COEFF_share_brown_slt",
]

eulfs_visualiser.cols_to_calc_all = cols_to_aggregate
eulfs_visualiser.cols_to_calc = cols_to_calc

#### Plot (by industry)

In [ ]:
eulfs_visualiser.create_industry_boxplots(
    year=2019,
    cols_to_aggregate=cols_to_aggregate,
    cols_to_plot=cols_to_plot,
    grouping=["NACE1D_label", "COUNTRYW"],
    out_dir=os.path.join(
        useful_paths.figure_dir,
        "03_eulfs",
        "{year}",
        "employment_shares_by_industry",
    ),
)

In [ ]:
eulfs_visualiser.create_industry_boxplots(
    year=2019,
    cols_to_aggregate=cols_to_aggregate,
    cols_to_plot=cols_to_plot,
    grouping=["NACE1D_label"],
    out_dir=os.path.join(
        useful_paths.figure_dir,
        "03_eulfs",
        "{year}",
        "employment_shares_by_industry",
        "grouped_by_industry_only",
    ),
)

#### Plot (by region)

In [ ]:
# sns.set_style("ticks")
# sns.set_context("paper", font_scale=1.5)
#
# perc_max = 3
# eulfs_visualiser.create_maps(
#     slice_by_industry=False,
#     vmin=0,
#     vmax=perc_max/100,
#     cmap="Oranges",
#     n_cats=perc_max*2,
#     out_dir=os.path.join(
#             useful_paths.figure_dir,
#             "03_eulfs",
#             "2019",
#             "employment_shares_by_region",
#             "NEW",
#             "cbars_fixed_at_{}_percent_discrete".format(perc_max)
#         ),
# )

In [ ]:
# eulfs_visualiser.create_maps(
#     slice_by_industry=True,
#     vmin=0,
#     vmax=None,
#     cmap="Greys",
#     n_cats=None,
#     out_dir=os.path.join(
#         useful_paths.figure_dir,
#         "03_eulfs",
#         "2019",
#         "employment_shares_by_region",
#         "NEW",
#         "by_industry",
#     ),
# )

Bar charts by country

In [ ]:
agg = {
    "NOBS": np.sum,
    "COEFF": np.sum,
    "COEFF_share_brown_sl": np.sum,
    "COEFF_share_brown_slt": np.sum,
    "COEFF_share_green": np.sum,
    "COEFF_share_neutral_sl": np.sum,
    "COEFF_share_neutral_slt": np.sum,
}

data_agg = lfs_data.groupby(["COUNTRYW", "NACE1D_label"]).aggregate(agg)

# calculate relative shares at regional level
for col in data_agg.columns[data_agg.columns.str.startswith("COEFF_")]:
    data_agg["{}_relative".format(col)] = data_agg[col] / data_agg["COEFF"]

In [ ]:
data_agg_final = data_agg[data_agg["NOBS"] >= threshold_a]

#### Plots by industry (w/o country grouping)

In [ ]:
df = lfs_data
for col in cols_to_aggregate:
    df["{}_relative".format(col)] = df[col] / df["COEFF"]

df

Diskussion mit Flo bzgl Medianen in diesem Plot:
- es macht Sinn, das 50% aller Jobs über alle Länder in einem der Sektoren total neutral sind
- deshalb sind die Mediane auch grossteils gleich null

In [ ]:
x = "COEFF_share_green_relative"
y = "NACE1D_label"

ax = sns.boxplot(
    data=df,
    x=x,
    y=y,
    order=df.groupby(y)[x].mean().sort_values(ascending=False).index.values,
    color=".5",
    fliersize=1,
    showfliers=True,
)

ax.set_xlim(0, 0.005)

df.groupby(y)[x].median()

Bar plots to also show absolute numbers

Our estimate of 212.5 Million compares well with the [offical size of labour force](https://data.worldbank.org/indicator/SL.TLF.TOTL.IN?locations=EU), 217.8 Million (some countries are missing in our estimates).

In [ ]:
df.COEFF.sum()

##### Absolute/relative COEFF by industry and scenario
- https://stackoverflow.com/questions/72936696/plotting-ax-bar-label-on-sns-does-not-iterate-through-hue-appropriately

In [ ]:
x_vars = [
    "COEFF_share_coal",
    "COEFF_share_brown_slt",
    "COEFF_share_brown_sl",
    "COEFF_share_green",
]

new_labels = ["CPO", "FFPOTC", "FFPO", "Green"]

relabel = dict(zip(x_vars, new_labels))
df = df.rename(columns=relabel)

In [ ]:
df

In [ ]:
df_long = pd.melt(
    df,
    value_vars=new_labels,
    id_vars=set(df.columns.values) - set(new_labels),
    var_name="Pool",
    value_name="Workers",
)

In [ ]:
df_long

In [ ]:
assert 1 == 2

In [ ]:
df_long.replace({"Pool": {"FFPOTC": "Brown"}})

##### Sectoral distribution (2C)

In [ ]:
from src.plotting_utils import move_legend_to_right, move_legend_to_top
sns.set_style("ticks")
sns.set_context("paper", font_scale=1.5)

def estimator(x):
    return np.sum(x) / 1000

# params
x = "Workers"
y = "NACE1D_label_short"
hue = "Pool"
hue_order = ["Brown", "Green"]
dpi = 300
shift_ylabels_right = True
figsize = (5, 8) if not shift_ylabels_right else (3, 8)

df_long = df_long.replace({"Pool": {"FFPOTC": "Brown"}})

# cmap
color_mapping = {
    # "FFPO": "saddlebrown",
    "Brown": "sandybrown",
    # "CPO": "peachpuff",
    "Green": "yellowgreen",
}
cmap = mpl.colors.ListedColormap(colors=[color_mapping[pool] for pool in hue_order])
mpl.cm.register_cmap("mycolormap", cmap)
cpal = sns.color_palette("mycolormap", n_colors=len(color_mapping), desat=1)

# plot
for industry_subset in [True, False]:

    if industry_subset:
        df_sub = df.loc[(df["paper_selection"])]
        df_long_sub = df_long.replace({"Pool": {"FFPOTC": "Brown"}}).loc[(df_long["paper_selection"]) & df_long[hue].isin(hue_order)]

        fontsize = 8
        rotation = 0
        logscale = False
    else:
        df_sub = df
        df_long_sub = df_long.replace({"Pool": {"FFPOTC": "Brown"}})
        logscale = True

        fontsize = 5
        rotation = 90

    industry_order = (
        df_sub.groupby(y)["FFPOTC"].sum().sort_values(ascending=False).index.values
    )
    print(df_long_sub[hue].unique())
    # plot
    fig, ax = plt.subplots(figsize=figsize)
    sns.barplot(
        data=df_long_sub,
        x=x,
        y=y,
        hue=hue,
        hue_order=hue_order,
        order=industry_order,
        estimator=estimator,
        errorbar=None,
        width=0.5,
        palette=cpal,
        ax=ax,
    )

    # bar labels
    sum_across_cnt_share = df_long_sub.groupby([y, hue])[x].sum()
    sum_across_cnt = df_long_sub.groupby([y, hue])["COEFF"].sum()
    ind_share = sum_across_cnt_share.divide(sum_across_cnt)
    ind_share.name = "employment_relative"
    ind_share = ind_share.reindex(industry_order, level=0)

    df_ind_share = ind_share.reset_index()
    df_ind_share["employment_absolute"] = sum_across_cnt_share.reindex(
        industry_order, level=0
    ).values
    df_ind_share["employment_relative_pc"] = (
        df_ind_share["employment_relative"].multiply(100).round(1)
    )
    df_ind_share["employment_relative_pc_labels"] = [
        "{}%".format(l) for l in df_ind_share["employment_relative_pc"]
    ]

    # one container per hue level
    for container in ax.containers:
        df_hue = df_ind_share.loc[df_ind_share[hue] == container.get_label()]
        ax.bar_label(
            container, labels=df_hue["employment_relative_pc_labels"], fontsize=fontsize
        )

    # layout
    if logscale:
        ax.set_xscale("log")
        ax.set_xlabel("Number of Workers, log [-]")
        plt.xlim(1, 10e3)
    else:
        ax.set_xlabel("Employment number [k]")
        ax.set_xlim(0, 1000)
        ax.set_xticks(np.arange(0, 1250, 250), major=True)
        #ax.set_xticks(np.arange(0, 1000, 125), major=False, labels=None)
        #ax.set_xticklabels(np.repeat(None, len(np.arange(0, 1000, 125))), minor=True)

    if shift_ylabels_right:
        # Create offset transform by X points in x direction
        dx = 50 / dpi
        dy = 100 / dpi
        offset = mpl.transforms.ScaledTranslation(dx, dy, fig.dpi_scale_trans)

        # apply offset transform to all x ticklabels.
        for label in ax.yaxis.get_majorticklabels():
            label.set_horizontalalignment("left")
            label.set_transform(label.get_transform() + offset)
            #label.set_fontsize(8)

    # grid and labelling
    ax.grid(axis="x", which="major", linestyle="--", zorder=0)
    ax.set_ylabel(None)
    plt.xticks(rotation=rotation)

    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles=handles, labels=labels, fancybox=False, frameon=False)

    sns.despine()
    plt.tight_layout()

    if industry_subset:
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles=handles, labels=labels, fancybox=False, frameon=False, bbox_to_anchor=(1, 1.08), ncol=2, fontsize=12)

    # save
    out_path = os.path.join(
        useful_paths.figure_dir,
        "03_eulfs",
        "2019",
        "employment_shares_by_industry",
        "grouped_by_industry_only",
        "{}_by_{}_scenarios_subset_{}_paper.{}",
    )
    plt.savefig(
        out_path.format(x, y, industry_subset, "png"),
        dpi=dpi,
        bbox_inches="tight",
    )
    df_ind_share.to_csv(
        out_path.format(x, y, industry_subset, "csv"),
    )

In [ ]:
industry_order

In [ ]:
from src.plotting_utils import move_legend_to_right, move_legend_to_top

# params
x = "Workers"
y = "NACE1D_label_short"
hue = "Pool"
hue_order = ["Green", "FFPO", "FFPOTC", "CPO"]
dpi = 300
shift_ylabels_right = True
figsize = (5, 8) if not shift_ylabels_right else (3, 8)

# cmap
color_mapping = {
    "FFPO": "saddlebrown",
    "FFPOTC": "sandybrown",
    "CPO": "peachpuff",
    "Green": "yellowgreen",
}

cmap = mpl.colors.ListedColormap(colors=[color_mapping[pool] for pool in hue_order])

mpl.cm.register_cmap("mycolormap", cmap)
cpal = sns.color_palette("mycolormap", n_colors=4, desat=1)

# plot
for industry_subset in [True, False]:

    if industry_subset:
        df_sub = df.loc[df["paper_selection"]]
        df_long_sub = df_long.loc[df_long["paper_selection"]]

        fontsize = 8
        rotation = 0
        logscale = False
    else:
        df_sub = df
        df_long_sub = df_long
        logscale = True

        fontsize = 5
        rotation = 90

    industry_order = (
        df_sub.groupby(y)["FFPO"].sum().sort_values(ascending=False).index.values
    )

    # plot
    fig, ax = plt.subplots(figsize=figsize)
    sns.barplot(
        data=df_long_sub,
        x=x,
        y=y,
        hue=hue,
        hue_order=hue_order,
        order=industry_order,
        estimator=np.sum,
        ci=None,
        # width=0.8,
        palette=cpal,
        ax=ax,
    )

    # bar labels
    sum_across_cnt_share = df_long_sub.groupby([y, hue])[x].sum()
    sum_across_cnt = df_long_sub.groupby([y, hue])["COEFF"].sum()
    ind_share = sum_across_cnt_share.divide(sum_across_cnt)
    ind_share.name = "employment_relative"
    ind_share = ind_share.reindex(industry_order, level=0)

    df_ind_share = ind_share.reset_index()
    df_ind_share["employment_absolute"] = sum_across_cnt_share.reindex(
        industry_order, level=0
    ).values
    df_ind_share["employment_relative_pc"] = (
        df_ind_share["employment_relative"].multiply(100).round(1)
    )
    df_ind_share["employment_relative_pc_labels"] = [
        "{}%".format(l) for l in df_ind_share["employment_relative_pc"]
    ]

    # one container per hue level
    for container in ax.containers:
        df_hue = df_ind_share.loc[df_ind_share[hue] == container.get_label()]
        ax.bar_label(
            container, labels=df_hue["employment_relative_pc_labels"], fontsize=fontsize
        )

    # layout
    if logscale:
        ax.set_xscale("log")
        ax.set_xlabel("Number of Workers, log [-]")
        plt.xlim(1, 10e6)
    else:
        ax.set_xlabel("Number of Workers [-]")

    if shift_ylabels_right:
        # Create offset transform by X points in x direction
        dx = 250 / dpi
        dy = 100 / dpi
        offset = mpl.transforms.ScaledTranslation(dx, dy, fig.dpi_scale_trans)

        # apply offset transform to all x ticklabels.
        for label in ax.yaxis.get_majorticklabels():
            label.set_horizontalalignment("left")
            label.set_transform(label.get_transform() + offset)

    # grid and labelling
    ax.grid(axis="x", which="major", linestyle="--", zorder=0)
    ax.set_ylabel(None)
    plt.xticks(rotation=rotation)

    sns.despine()
    plt.tight_layout()

    if industry_subset:
        move_legend_to_top(ax, ncol=2, lower_anchor=0.5, upper_anchor=1.12)

    # save
    out_path = os.path.join(
        useful_paths.figure_dir,
        "03_eulfs",
        "2019",
        "employment_shares_by_industry",
        "grouped_by_industry_only",
        "{}_by_{}_scenarios_subset_{}.{}",
    )
    plt.savefig(
        out_path.format(x, y, industry_subset, "png"),
        dpi=dpi,
        bbox_inches="tight",
    )
    df_ind_share.to_csv(
        out_path.format(x, y, industry_subset, "csv"),
    )

In [ ]:
pd.options.display.float_format = "{:,.2f}".format
df_ind_share

##### Absolute/relative COEFF by industry

In [ ]:
y = "NACE1D_label_short"

for x in new_labels:
    f, ax = plt.subplots(figsize=(10, 7))
    sns.barplot(
        data=df,
        x=x,
        y=y,
        order=df.groupby(y)[x].sum().sort_values(ascending=False).index.values,
        estimator=np.sum,
        ci=None,
        palette="viridis_r",
        ax=ax,
    )

    # calculate sector-level percentage
    order = df.groupby(y)[x].sum().sort_values(ascending=False).index
    sum_across_cnt_share = df.groupby(y)[x].sum()
    sum_across_cnt = df.groupby(y)["COEFF"].sum()
    ind_share = sum_across_cnt_share.divide(sum_across_cnt)
    percentages = ind_share.reindex(order).multiply(100).round(1)
    percent_labels = ["{}%".format(l) for l in percentages]

    # show the percentage within sector
    ax.bar_label(ax.containers[0], labels=percent_labels, padding=1)
    ax.set_ylabel(None)

    ax.grid(axis="x", linestyle="--", zorder=0)
    sns.despine()
    plt.tight_layout()
    plt.savefig(
        os.path.join(
            useful_paths.figure_dir,
            "03_eulfs",
            "2019",
            "employment_shares_by_industry",
            "grouped_by_industry_only",
            "{}_by_{}_absolute.png".format(x, y),
        ),
        dpi=150,
    )
    # plt.clf()

Absolute/relative COEFF by country (OLD plot)

In [ ]:
# params
x = "Workers"
y = "COUNTRYW"
hue = "Pool"

industry_order = df.groupby(y)["FFPO"].sum().sort_values(ascending=False).index.values

# plot
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(
    data=df_long,
    x=y,
    y=x,
    hue=hue,
    order=industry_order,
    estimator=np.sum,
    ci=None,
    palette=cpal,
    ax=ax,
)

# bar labels
sum_across_cnt_share = df_long.groupby([y, hue])[x].sum()
sum_across_cnt = df_long.groupby([y, hue])["COEFF"].sum()
cnt_share = sum_across_cnt_share.divide(sum_across_cnt).multiply(100)
cnt_share.name = "employment_relative"
cnt_share = cnt_share.reindex(industry_order, level=0)
sum_across_cnt_share = sum_across_cnt_share.reindex(industry_order, level=0)

df_cnt_share = cnt_share.reset_index()
df_cnt_share["percentages"] = df_cnt_share["employment_relative"].round(1)
df_cnt_share["percent_labels"] = ["{}".format(l) for l in df_cnt_share["percentages"]]
df_cnt_share["employment_absolute"] = sum_across_cnt_share.values

# one container per hue level
for container in ax.containers:
    df_hue = df_cnt_share.loc[df_cnt_share[hue] == container.get_label()]
    ax.bar_label(container, labels=df_hue["percent_labels"], fontsize=5)

# layout
ax.set_yscale("log")
ax.grid(axis="y", linestyle="--", zorder=0)

ax.set_ylabel("Number of Workers, log [-]")
ax.set_xlabel(None)

# plt.yticks(rotation=90)
plt.ylim(1, 10e6)
sns.despine()

# save
plt.tight_layout()
plt.savefig(
    os.path.join(
        useful_paths.figure_dir,
        "03_eulfs",
        "2019",
        "employment_shares_by_country",
        "{}_by_{}_scenarios.png".format(x, y),
    ),
    dpi=300,
)

##### Quadrant style: abs vs rel impact

##### Version 1 (With Green)

In [ ]:
df_cnt_share.drop(columns=["percentages", "percent_labels"]).to_csv(
    os.path.join(
        useful_paths.figure_dir,
        "03_eulfs",
        "2019",
        "employment_shares_by_country",
        "employment_abs_rel_by_country.csv",
    )
)

In [ ]:
df_cnt_share["employment_absolute_mio"] = df_cnt_share["employment_absolute"] / 10**6
country_means = df_cnt_share.groupby("Pool").mean()

In [ ]:
df_cnt_share

In [ ]:
df_cnt_share

##### Version 1: FFPOTC and Green

In [ ]:
facet_kws = {"sharey": False, "sharex": True}
col_order = ["Brown", "Green"]
fill_alpha = 0.1

x = "employment_absolute_mio"
y = "percentages"
data = df_cnt_share.loc[df_cnt_share["Pool"].isin(col_order)]

g = sns.relplot(
    data=data,
    x=x,
    y=y,
    col="Pool",
    col_order=col_order,
    #hue="COUNTRYW",
    style="COUNTRYW",
    kind="scatter",
    color="grey",
    col_wrap=2,
    s=100,
    facet_kws=facet_kws,
    legend=None
)

# annotate
for i, (ax, pool) in enumerate(zip(g.axes, col_order)):

    # draw means
    x_mean = country_means.loc[pool, x]
    y_mean = country_means.loc[pool, y]

    ax.axvline(x_mean, color="grey", linestyle=":")
    ax.axhline(y_mean, color="grey", linestyle=":")

    # flip colors between brown/green
    if pool == "Brown":
        cats = {
            "low": "green",
            "mid": "orange",
            "high": "red"
        }

        xmax = 1
        ymax = 1.75
    elif pool == "Green":
        cats = {
            "low": "red",
            "mid": "orange",
            "high": "green"
        }
        xmax = 1
        ymax = 3.5

    # low-risk
    ax.fill_between(
        x=[0, x_mean], y1=0, y2=y_mean, color=cats["low"], alpha=fill_alpha, zorder=0
    )

    # high-risk
    ax.fill_between(
        x=[x_mean, xmax],
        y1=y_mean,
        y2=ymax,
        color=cats["high"],
        alpha=fill_alpha,
        zorder=0,
    )

    # mid-risk
    ax.fill_between(
        x=[0, x_mean],
        y1=y_mean,
        y2=ymax,
        color=cats["mid"],
        alpha=fill_alpha,
        zorder=0,
    )
    ax.fill_between(
        x=[x_mean, xmax],
        y1=0,
        y2=y_mean,
        color=cats["mid"],
        alpha=fill_alpha,
        zorder=0,
    )

    # country codes
    offset = 0.01
    data_sub = data.loc[data["Pool"] == pool]
    for _, s in data_sub.iterrows():
        ax.annotate(
            text=s["COUNTRYW"],
            xy=(s[x], s[y]),
            xytext=(s[x]+s[x]*offset, s[y]+s[y]*offset)
        )

    # ticks
    # ax.set_xticks(np.arange(0, 1.2, 0.2), minor=False)
    # ax.set_xticks(np.arange(0, 1.1, 0.1), minor=True)
    #
    # ax.set_yticks(np.arange(0, 5, 1), minor=False)
    # ax.set_yticks(np.arange(0, 4, 0.5), minor=True)
    #
    # ax.set_xlim(-.5, 1)
    # ax.set_ylim(0, 4)

    # labels
    ax.set_xlabel("Employment number (mn)")
    if i == 0:
        ax.set_ylabel("Employment share (%)")

# sns.move_legend(g, loc="center right", ncol=2, bbox_to_anchor=(1.03, 0.5), frameon=True)
# plt.tight_layout()

# save
plt.savefig(
    os.path.join(
        useful_paths.figure_dir,
        "03_eulfs",
        "2019",
        "employment_shares_by_country",
        "transition_risk_indicator_short_paper.png",
    ),
    dpi=300,
    bbox_inches="tight",
)


In [ ]:
assert 1 == 2

##### Version 2 (w/o Green)

In [ ]:
facet_kws = {"sharey": False, "sharex": False}
col_order = ["CPO", "FFPOTC", "FFPO"]
fill_alpha = 0.1

g = sns.relplot(
    data=df_cnt_share.loc[df_cnt_share["Pool"] != "Green"],
    x="employment_absolute",
    y="percentages",
    col="Pool",
    col_order=col_order,
    hue="COUNTRYW",
    style="COUNTRYW",
    palette="Paired",
    kind="scatter",
    col_wrap=3,
    s=100,
    facet_kws=facet_kws,
)

# annotate means
for i, (ax, pool) in enumerate(zip(g.axes, col_order)):

    # draw means
    x_mean = country_means.loc[pool, "employment_absolute"]
    y_mean = country_means.loc[pool, "percentages"]

    ax.axvline(x_mean, color="grey", linestyle=":")
    ax.axhline(y_mean, color="grey", linestyle=":")

    # fill areas
    x = df_cnt_share.loc[df_cnt_share["Pool"] == pool, "employment_absolute"]
    y = df_cnt_share.loc[df_cnt_share["Pool"] == pool, "percentages"]

    # low-risk
    ax.fill_between(
        x=[0, x_mean], y1=0, y2=y_mean, color="green", alpha=fill_alpha, zorder=0
    )
    # high-risk
    ax.fill_between(
        x=[x_mean, np.max(x)],
        y1=y_mean,
        y2=np.max(y),
        color="red",
        alpha=fill_alpha,
        zorder=0,
    )
    # mid-risk
    ax.fill_between(
        x=[0, x_mean],
        y1=y_mean,
        y2=np.max(y),
        color="orange",
        alpha=fill_alpha,
        zorder=0,
    )
    ax.fill_between(
        x=[x_mean, np.max(x)],
        y1=0,
        y2=y_mean,
        color="orange",
        alpha=fill_alpha,
        zorder=0,
    )

    # labels
    ax.set_xlabel("Employment (-)")
    if i == 0:
        ax.set_ylabel("Employment share (%)")

# sns.move_legend(g, loc="center right", ncol=2, bbox_to_anchor=(1.03, 0.5), frameon=True)
# plt.tight_layout()

# save
plt.savefig(
    os.path.join(
        useful_paths.figure_dir,
        "03_eulfs",
        "2019",
        "employment_shares_by_country",
        "transition_risk_indicator_short.png",
    ),
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
y = "COUNTRYW"

for x in new_labels:
    f, ax = plt.subplots(figsize=(10, 7))
    sns.barplot(
        data=df,
        x=y,
        y=x,
        order=df.groupby(y)[x].sum().sort_values(ascending=False).index.values,
        estimator=np.sum,
        ci=None,
        color="Grey",
    )

    # calculate country-level percentage
    sum_across_cnt_share = df.groupby(y)[x].sum()
    sum_across_cnt = df.groupby(y)["COEFF"].sum()
    ind_share = sum_across_cnt_share.divide(sum_across_cnt)
    order = sum_across_cnt_share.sort_values(ascending=False).index
    percentages = ind_share.reindex(order).multiply(100).round(1)
    percent_labels = ["{}%".format(l) for l in percentages]

    # show the percentage within sector
    ax.bar_label(ax.containers[0], labels=percent_labels, padding=1)

    ax.ticklabel_format(useOffset=False, style="plain", axis="y")
    ax.grid(axis="y", linestyle="--", zorder=0)
    sns.despine()
    plt.tight_layout()
    plt.savefig(
        os.path.join(
            useful_paths.figure_dir,
            "03_eulfs",
            "2019",
            "employment_shares_by_country",
            "{}_by_{}_absolute.png".format(x, y),
        ),
        dpi=150,
    )
    plt.close(f)

Absolute/relative COEFF at EU level

In [ ]:
res = []
for x in new_labels:
    sum_total = df[x].sum()
    print(x, sum_total, 100 * (sum_total / df["COEFF"].sum()))
    res.append([x, sum_total])

df_res = pd.DataFrame(res, columns=["share_type", "employment"])
df_res = df_res.set_index("share_type")
ax = (
    df_res.iloc[0:3, :]
    .sort_values(by="employment", ascending=False)
    .plot.bar(color="grey")
)

ax.grid(axis="y", linestyle="--", zorder=0)
sns.despine()
plt.tight_layout()

plt.savefig(
    os.path.join(
        useful_paths.figure_dir,
        "03_eulfs",
        "2019",
        "employment_shares_by_country",
        "EU_employment_by_share_type_absolute.png".format(x, y),
    ),
    dpi=300,
)

df_res.to_csv(
    os.path.join(
        useful_paths.figure_dir,
        "03_eulfs",
        "2019",
        "employment_shares_by_country",
        "EU_employment_by_share_type_absolute.csv",
    ),
)

In [ ]:
facet_kws = {"sharey": False, "sharex": False}
col_order = ["CPO", "FFPOTC", "FFPO", "Green"]

g = sns.relplot(
    data=df_cnt_share,
    x="employment_absolute",
    y="employment_relative",
    col="Pool",
    col_order=col_order,
    hue="COUNTRYW",
    style="COUNTRYW",
    palette="Paired",
    kind="scatter",
    col_wrap=2,
    s=100,
    facet_kws=facet_kws,
)

# annotate means
for ax, pool in zip(g.axes, col_order):
    x_mean = country_means.loc[pool, "employment_absolute"]
    y_mean = country_means.loc[pool, "employment_relative"]

    ax.axvline(x_mean, color="grey", linestyle=":")
    ax.axhline(y_mean, color="grey", linestyle=":")

# save
out_path = os.path.join(
    useful_paths.figure_dir,
    "03_eulfs",
    "2019",
    "employment_shares_by_country",
    "transition_risk_indicator.{}",
)
plt.savefig(
    out_path.format("png"),
    dpi=300,
    bbox_inches="tight",
)

df_cnt_share.to_csv(out_path.format("csv"))

In [ ]:
df_res_sl = df_res.loc[(new_labels), :]
df_res_sl.divide(df_res_sl.sum()) * 100

#### Shares by sociodemographic variables

Degree of urbanisation (DEGURBA)

1 Cities (Densely-populated area)
2 Towns and suburbs (Intermediate density area)
3 Rural area (Thinly-populated area)

Gender (SEX)

1 Male
2 Female

Level of education
L Low: Lower secondary
M Medium: Upper secondary
H High: Third level

In [ ]:
y = "SEX"

for x in new_labels:
    f, ax = plt.subplots(figsize=(10, 7))
    sns.barplot(data=df, x=y, y=x, order=None, estimator=np.sum, ci=None, color="Grey")

    # calculate country-level percentage
    sum_across_cnt_share = df.groupby(y)[x].sum()
    sum_across_cnt = df.groupby(y)["COEFF"].sum()
    ind_share = sum_across_cnt_share.divide(sum_across_cnt)
    order = sum_across_cnt_share.sort_values(ascending=False).index
    percentages = ind_share.reindex(order).multiply(100).round(1)
    percent_labels = ["{}%".format(l) for l in percentages]

    # show the percentage within sector
    ax.bar_label(ax.containers[0], labels=percent_labels, padding=1)

    ax.ticklabel_format(useOffset=False, style="plain", axis="y")
    ax.grid(axis="y", linestyle="--", zorder=0)
    sns.despine()
    plt.tight_layout()
    out_dir = os.path.join(
        useful_paths.figure_dir, "03_eulfs", "2019", "employment_by_{}".format(y)
    )
    utils.ccdir(out_dir)
    fname = "{}_by_{}_absolute.png".format(x, y)
    plt.savefig(os.path.join(out_dir, fname), dpi=150)
    plt.close(f)

In [ ]:
meanprops = {
    "marker": "o",
    "markerfacecolor": "white",
    "markeredgecolor": "black",
    "markersize": "3",
}

In [ ]:
f, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(
    data=df,
    x="COUNTRYW",
    y="annual_earnings",
    hue="category_sl",
    fliersize=1,
    showmeans=True,
    meanprops=meanprops,
    ax=ax,
)

ax.grid(axis="y", linestyle="--", zorder=0)
ax.set_ylabel("Annual earnings [€ (2019)]")
ax.set_xlabel(None)

sns.despine()

In [ ]:
f, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(
    data=df,
    x="COUNTRYW",
    y="annual_earnings",
    hue="category_slt",
    fliersize=1,
    showmeans=True,
    meanprops=meanprops,
    ax=ax,
)

ax.grid(axis="y", linestyle="--", zorder=0)
ax.set_ylabel("Annual earnings [€ (2019)]")
ax.set_xlabel(None)

sns.despine()

In [ ]:
def weighted_mean(x):
    np.average(x, weights=x.COEFF)


# df.groupby("DEGURBA").aggregate({"share_brown_sl": weighted_mean})

In [ ]:
x_vars = ["share_green", "share_brown_sl", "share_brown_slt"]
y = "NACE1D_label"
hues = ["HATLEV1D", "DEGURBA", "SEX"]
hue_orders = [["H", "M", "L"], ["1", "2", "3"], ["1", "2"]]

for x in x_vars:
    for hue, hue_order in zip(hues, hue_orders):
        f, ax = plt.subplots(figsize=(10, 7))
        sns.pointplot(
            data=df,
            x=x,
            y=y,
            hue=hue,
            hue_order=hue_order,
            order=df.groupby(y)[x].mean().sort_values(ascending=False).index.values,
            color=".5",
            ci="sd",
            join=False,
            dodge=0.5,
            errwidth=1,
            # capsize=0.5,
            scale=0.7,
        )

        ax.set_xlim(0, 0.7)
        ax.grid(axis="x", linestyle="--", zorder=0)
        sns.despine()

        plt.tight_layout()
        plt.savefig(
            os.path.join(
                useful_paths.figure_dir,
                "03_eulfs",
                "2019",
                "employment_shares_by_industry",
                "grouped_by_industry_only",
                "{}_by_{}_and_{}.png".format(x, y, hue),
            ),
            dpi=150,
        )
        plt.close(f)

NOTE:
`share_green` precisely equals `COEFF_share_green_relative`

In [ ]:
np.isclose(df.share_green, df.COEFF_share_green_relative).all()

In [ ]:
df.groupby("DEGURBA").describe()

In [ ]:
group_var = "DEGURBA"
num_var = "share_brown_sl"
# sns.boxplot(x=df[group_var].astype(int), y=df[num_var])
sns.pointplot(
    x=df[group_var].astype(int),
    y=df[num_var],
    hue=df["HATLEV1D"].astype(str),
    ci="sd",
    dodge=True,
)

In [ ]:
x = "HATLEV1D"
y = "share_green"
hue = "SEX"
order = ["L", "M", "H"]
df2 = df.dropna(subset=[x])
sns.pointplot(
    x=df2[x].astype(str),
    y=df2[y],
    hue=df2[hue],
    dodge=True,
    palette="Greys",
    ci="sd",
    order=order,
)